In [ ]:
# ==========================================================
# PROCESSAMENTO DOS BOLETINS PAS
# ==========================================================

from pathlib import Path


def resolver_base() -> Path:
    """Resolve o diretório raiz do projeto em qualquer ambiente."""
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "tsv").exists() and (candidate / "catalogos").exists():
            return candidate

    return cwd


BASE = resolver_base()

# ==========================================================
# BIBLIOTECA
# ==========================================================

%run "./02_biblioteca_normalizacao_pas.ipynb"

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BIBLIOTECA CARREGADA
Triênios............. 6
Modalidades.......... 18
Cursos canônicos..... 90
Variantes............ 132


In [3]:
print(LAYOUTS.keys())

dict_keys(['2018-2020', '2019-2021', '2020-2022', '2021-2023', '2022-2024', '2023-2025'])


In [4]:
# ==========================================================
# PROCESSAMENTO
# ==========================================================

TRIENIOS = [

    "2018-2020",
    "2019-2021",
    "2020-2022",
    "2021-2023",
    "2022-2024",
    "2023-2025",

]

resultados = {}

print(LAYOUTS)

for trienio in TRIENIOS:

    print()

    print("=" * 70)
    print(trienio)
    print("=" * 70)

    resultados[trienio] = executar(trienio)

    # ======================================================
    # EXPORTAÇÃO DO DATAFRAME NORMALIZADO
    # ======================================================

    ARQUIVO_DF = BASE / "diagnosticos" / f"{trienio}.csv"

    ARQUIVO_DF.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    resultados[trienio].to_csv(
        ARQUIVO_DF,
        index=False,
        encoding="utf-8-sig"
    )

    # ======================================================
    # EXPORTAÇÃO DO CSV DE AUDITORIA
    # ======================================================

    ARQUIVO_AUDITORIA = (
        BASE
        / "diagnosticos"
        / f"{trienio}_auditoria.csv"
    )

    resultados[trienio].to_csv(
        ARQUIVO_AUDITORIA,
        index=False,
        encoding="utf-8-sig"
    )

    # ======================================================
    # EXPORTAÇÃO DO RESUMO JSON
    # ======================================================

    df = resultados[trienio]

    resumo_json = {

        "subprograma": trienio,

        "shape": list(df.shape),

        "registros": len(df),

        "campi": int(df["Campus"].nunique()),

        "cursos": int(df["Curso"].nunique()),

        "modalidades": int(df["Modalidade"].nunique()),

        "nota_min_global": float(df["Nota Mínima"].min()),

        "nota_max_global": float(df["Nota Máxima"].max()),

        "campi_contagem": (
            df["Campus"]
            .value_counts()
            .sort_index()
            .to_dict()
        ),

        "modalidades_contagem": (
            df["Modalidade"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

    }

    ARQUIVO_JSON = (
        BASE
        / "diagnosticos"
        / f"{trienio}_resumo.json"
    )

    with open(
        ARQUIVO_JSON,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            resumo_json,
            f,
            ensure_ascii=False,
            indent=4
        )

    print(f"DataFrame exportado: {ARQUIVO_DF}")
    print(f"Auditoria exportada: {ARQUIVO_AUDITORIA}")
    print(f"Resumo exportado...: {ARQUIVO_JSON}")

{'2018-2020': {'header': 4, 'tipo': 1, 'curso_col': 0, 'turno_col': 1, 'campus_col': None, 'campus_prefixo': 'Campus UnB —', 'modalidades': {(2, 3): 'EP_R1_PPI', (4, 5): 'EP_R1_PPI_PCD', (6, 7): 'EP_R1_NPPI', (8, 9): 'EP_R1_NPPI_PCD', (10, 11): 'EP_R2_PPI', (12, 13): 'EP_R2_PPI_PCD', (14, 15): 'EP_R2_NPPI', (16, 17): 'EP_R2_NPPI_PCD', (18, 19): 'CN', (20, 21): 'AC'}}, '2019-2021': {'header': 4, 'tipo': 2, 'campus_col': 0, 'turno_col': 1, 'curso_col': 2, 'campus_prefixo': None, 'modalidades': {(3, 4): 'CN', (5, 6): 'EP_R1_PPI', (7, 8): 'EP_R1_PPI_PCD', (9, 10): 'EP_R1_NPPI', (11, 12): 'EP_R1_NPPI_PCD', (13, 14): 'EP_R2_PPI', (15, 16): 'EP_R2_PPI_PCD', (17, 18): 'EP_R2_NPPI', (19, 20): 'EP_R2_NPPI_PCD', (21, 22): 'AC'}}, '2020-2022': {'header': 4, 'tipo': 1, 'curso_col': 0, 'turno_col': 1, 'campus_col': None, 'campus_prefixo': 'Campus UnB — ', 'modalidades': {(2, 3): 'EP_R1_PPI', (4, 5): 'EP_R1_PPI_PCD', (6, 7): 'EP_R1_NPPI', (8, 9): 'EP_R1_NPPI_PCD', (10, 11): 'EP_R2_PPI', (12, 13): 'EP

,Subprograma,Ano,Campus,Curso,Turno,Modalidade,modalidade_normalizada,escola_publica,faixa_renda,grupo_etnico,pcd,limite_salario_minimo,Nota Mínima,Nota Máxima
0,2018-2020,2020,Ceilândia,Terapia Ocupacional (Bacharelado),Diurno,EP_R1_NPPI,EP_R1_NPPI,True,R1,NPPI,False,1.5,-92.652,-4.201
1,2018-2020,2020,Ceilândia,Terapia Ocupacional (Bacharelado),Diurno,EP_R2_PPI,EP_R2_PPI,True,R2,PPI,False,1.5,-45.148,-25.927
2,2018-2020,2020,Ceilândia,Terapia Ocupacional (Bacharelado),Diurno,EP_R2_NPPI,EP_R2_NPPI,True,R2,NPPI,False,1.5,-74.630,27.307
3,2018-2020,2020,Ceilândia,Terapia Ocupacional (Bacharelado),Diurno,CN,CN,False,None,NEGROS,False,NaN,5.337,6.491
4,2018-2020,2020,Ceilândia,Terapia Ocupacional (Bacharelado),Diurno,AC,AC,False,None,None,False,NaN,-73.944,20.377



Arquivos gerados:
 • 2018-2020.csv
 • 2018-2020.json
DataFrame exportado: /content/drive/MyDrive/Dados/PAS/diagnosticos/2018-2020.csv
Auditoria exportada: /content/drive/MyDrive/Dados/PAS/diagnosticos/2018-2020_auditoria.csv
Resumo exportado...: /content/drive/MyDrive/Dados/PAS/diagnosticos/2018-2020_resumo.json

2019-2021
2019-2021
DIAGNÓSTICO
Registros..... 463
Campi......... 4
Cursos........ 85
Turnos........ 2
Modalidades... 8



,Subprograma,Ano,Campus,Curso,Turno,Modalidade,modalidade_normalizada,escola_publica,faixa_renda,grupo_etnico,pcd,limite_salario_minimo,Nota Mínima,Nota Máxima
0,2019-2021,2021,Darcy Ribeiro,Administração (Bacharelado),Diurno,CN,CN,False,None,NEGROS,False,NaN,-5.786,-0.285
1,2019-2021,2021,Darcy Ribeiro,Administração (Bacharelado),Diurno,EP_R1_PPI,EP_R1_PPI,True,R1,PPI,False,1.5,-38.327,23.576
2,2019-2021,2021,Darcy Ribeiro,Administração (Bacharelado),Diurno,EP_R1_NPPI,EP_R1_NPPI,True,R1,NPPI,False,1.5,-15.383,-0.009
3,2019-2021,2021,Darcy Ribeiro,Administração (Bacharelado),Diurno,EP_R2_PPI,EP_R2_PPI,True,R2,PPI,False,1.5,-32.915,0.643
4,2019-2021,2021,Darcy Ribeiro,Administração (Bacharelado),Diurno,EP_R2_NPPI,EP_R2_NPPI,True,R2,NPPI,False,1.5,-11.331,85.005



Arquivos gerados:
 • 2019-2021.csv
 • 2019-2021.json
DataFrame exportado: /content/drive/MyDrive/Dados/PAS/diagnosticos/2019-2021.csv
Auditoria exportada: /content/drive/MyDrive/Dados/PAS/diagnosticos/2019-2021_auditoria.csv
Resumo exportado...: /content/drive/MyDrive/Dados/PAS/diagnosticos/2019-2021_resumo.json

2020-2022
2020-2022
DIAGNÓSTICO
Registros..... 449
Campi......... 4
Cursos........ 88
Turnos........ 2
Modalidades... 8



...


Arquivos gerados:
 • 2021-2023.csv
 • 2021-2023.json
DataFrame exportado: /content/drive/MyDrive/Dados/PAS/diagnosticos/2021-2023.csv
Auditoria exportada: /content/drive/MyDrive/Dados/PAS/diagnosticos/2021-2023_auditoria.csv
Resumo exportado...: /content/drive/MyDrive/Dados/PAS/diagnosticos/2021-2023_resumo.json

2022-2024
2022-2024
DIAGNÓSTICO
Registros..... 168
Campi......... 3
Cursos........ 48
Turnos........ 2
Modalidades... 7



,Subprograma,Ano,Campus,Curso,Turno,Modalidade,modalidade_normalizada,escola_publica,faixa_renda,grupo_etnico,pcd,limite_salario_minimo,Nota Mínima,Nota Máxima
0,2022-2024,2024,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R1_NPPI,EP_R1_NPPI,True,R1,NPPI,False,1.5,-49.851,-32.128
1,2022-2024,2024,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R2_PPI,EP_R2_PPI,True,R2,PPI,False,1.5,-77.474,-39.613
2,2022-2024,2024,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R2_NPPI,EP_R2_NPPI,True,R2,NPPI,False,1.5,-16.271,-1.680
3,2022-2024,2024,Ceilândia,Enfermagem (Bacharelado),Diurno,CN,CN,False,None,NEGROS,False,NaN,-45.031,-45.031
4,2022-2024,2024,Ceilândia,Enfermagem (Bacharelado),Diurno,AC,AC,False,None,None,False,NaN,2.383,9.145



Arquivos gerados:
 • 2022-2024.csv
 • 2022-2024.json
DataFrame exportado: /content/drive/MyDrive/Dados/PAS/diagnosticos/2022-2024.csv
Auditoria exportada: /content/drive/MyDrive/Dados/PAS/diagnosticos/2022-2024_auditoria.csv
Resumo exportado...: /content/drive/MyDrive/Dados/PAS/diagnosticos/2022-2024_resumo.json

2023-2025
2023-2025
DIAGNÓSTICO
Registros..... 345
Campi......... 4
Cursos........ 85
Turnos........ 2
Modalidades... 9



,Subprograma,Ano,Campus,Curso,Turno,Modalidade,modalidade_normalizada,escola_publica,faixa_renda,grupo_etnico,pcd,limite_salario_minimo,Nota Mínima,Nota Máxima
0,2023-2025,2025,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R1_NPPIQ,EP_R1_NPPIQ,True,R1,NPPIQ,False,1.0,-46.839,3.143
1,2023-2025,2025,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R2_PPIQ,EP_R2_PPIQ,True,R2,PPIQ,False,1.0,-46.024,-16.220
2,2023-2025,2025,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R2_NPPIQ,EP_R2_NPPIQ,True,R2,NPPIQ,False,1.0,8.744,18.405
3,2023-2025,2025,Ceilândia,Enfermagem (Bacharelado),Diurno,EP_R2_NPPIQ_PCD,EP_R2_NPPIQ_PCD,True,R2,NPPIQ,True,1.0,-43.640,-43.640
4,2023-2025,2025,Ceilândia,Enfermagem (Bacharelado),Diurno,CN,CN,False,None,NEGROS,False,NaN,22.701,22.701



Arquivos gerados:
 • 2023-2025.csv
 • 2023-2025.json
DataFrame exportado: /content/drive/MyDrive/Dados/PAS/diagnosticos/2023-2025.csv
Auditoria exportada: /content/drive/MyDrive/Dados/PAS/diagnosticos/2023-2025_auditoria.csv
Resumo exportado...: /content/drive/MyDrive/Dados/PAS/diagnosticos/2023-2025_resumo.json


In [5]:
# ==========================================================
# RESUMO
# ==========================================================

resumo = []

for trienio, df in resultados.items():

    resumo.append({

        "Triênio": trienio,

        "Registros": len(df),

        "Campi": df["Campus"].nunique(),

        "Cursos": df["Curso"].nunique(),

        "Turnos": df["Turno"].nunique(),

        "Modalidades": df["Modalidade"].nunique(),

        "Nota mínima": df["Nota Mínima"].min(),

        "Nota máxima": df["Nota Máxima"].max()

    })

resumo = pd.DataFrame(resumo)

display(resumo)

,Triênio,Registros,Campi,Cursos,Turnos,Modalidades,Nota mínima,Nota máxima
0,2018-2020,436,4,88,2,8,-102.234,190.742
1,2019-2021,463,4,85,2,8,-98.760,192.190
2,2020-2022,449,4,88,2,8,-105.487,209.070
3,2021-2023,445,4,86,2,9,-96.102,211.738
4,2022-2024,168,3,48,2,7,-105.030,173.057
5,2023-2025,345,4,85,2,9,-100.426,190.334


In [6]:
# ==========================================================
# EXPORTAÇÃO DO DIAGNÓSTICO
# ==========================================================

ARQUIVO = BASE / "diagnosticos" / "resumo_processamento.csv"

ARQUIVO.parent.mkdir(
    parents=True,
    exist_ok=True
)

resumo.to_csv(
    ARQUIVO,
    index=False,
    encoding="utf-8-sig"
)

print()

print("=" * 70)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 70)

print()

print(resumo)

print()

print(f"Resumo salvo em:\n{ARQUIVO}")


PROCESSAMENTO CONCLUÍDO

     Triênio  Registros  Campi  Cursos  Turnos  Modalidades  Nota mínima  \
0  2018-2020        436      4      88       2            8     -102.234   
1  2019-2021        463      4      85       2            8      -98.760   
2  2020-2022        449      4      88       2            8     -105.487   
3  2021-2023        445      4      86       2            9      -96.102   
4  2022-2024        168      3      48       2            7     -105.030   
5  2023-2025        345      4      85       2            9     -100.426   

   Nota máxima  
0      190.742  
1      192.190  
2      209.070  
3      211.738  
4      173.057  
5      190.334  

Resumo salvo em:
/content/drive/MyDrive/Dados/PAS/diagnosticos/resumo_processamento.csv
